In [2]:
from functools import reduce 
import numpy as np
import matplotlib.pyplot as plt
import rioxarray
import rasterio
import glob
import os
import cv2
from scipy.interpolate import interp1d
from matplotlib.patches import Rectangle
import tqdm
from rasterio.transform import Affine
from shapely.geometry import Polygon, Point, LineString, MultiPolygon
import geopandas as gpd
import pandas as pd
import tqdm
from tqdm.auto import tqdm
from shapely.ops import unary_union
from shapely.geometry import box
import fiona
from osgeo import gdal, ogr
import geopandas as gpd

In [3]:
file1_path = "../Polygons_intersections/final_combined_polygons_2024_unmix.geojson"
output_file='./transformed_final_combined_polygons_2024_unmix.geojson'
#file2_path = "../Polygons_intersections/mesta_zarost_borch2023.geojson"

In [8]:
try:
    grid1 = gpd.read_file(file1_path)
    #grid2 = gpd.read_file(file2_path)
except Exception as e:
    print(f"Ошибка при загрузке файлов: {e}")
    exit()

In [5]:
grid1

,geometry
0,"POLYGON ((36.05079 54.3865, 36.0507 54.3865, 3..."
1,"POLYGON ((36.09467 54.38309, 36.09458 54.38309..."
2,"POLYGON ((36.07597 54.38204, 36.07587 54.38204..."
3,"POLYGON ((36.05194 54.3772, 36.0519 54.37725, ..."
4,"POLYGON ((35.98489 54.37007, 35.9848 54.37007,..."
...,...
21571,"POLYGON ((35.82378 55.83343, 35.82369 55.83343..."
21572,"POLYGON ((35.92416 55.83315, 35.92406 55.83315..."
21573,"POLYGON ((35.87047 55.83075, 35.87037 55.83075..."
21574,"POLYGON ((35.86965 55.82669, 35.86955 55.82669..."


In [9]:
#создаём эталон и целевые точки (координаты записаны в docx и взяты из final comb и сопоставллись с фото Yandex Sattelite)

#GeoJSON1 final_combine

src_coord=[(36.70223571724127,55.193600162463724), (36.703137301843135, 55.19312768929971), (36.70531740472222, 55.19513093702003), (36.70517478116939, 55.194630861394145) ] #исходные координаты
dst_coord=[(36.70226118573285, 55.19374553769437), (36.70320606677039, 55.19341844267927), (36.70546002827506, 55.19473552890516), (36.704991408030025, 55.19504952978788)]  #целевые коорд в геопривязке 3

#вычисляем афинное преобразование 

def calculate_affine_transformation(src_coord, dst_coord):
    
    n=4  #количество точек
    A=np.zeros((2*n,6))
    b=np.zeros(2*n)
    
    for i in range(n):
        x,y=src_coord[i]
        x_prime, y_prime = dst_coord[i]
        
        A[2*i,:] = [x,y,1,0,0,0]
        A[2*i+1,:] = [0,0,0,x,y,1] # заполняем коэффициенты a,b,c,d,e,f
        
        b[2*i]=x_prime
        b[2*i+1]=y_prime
    try: 
        transform, residuals, rank, s = np.linalg.lstsq(A,b,rcond=None) # решение методом наименьших квадратов(least Squares)
        '''
        Где A - матрица коэффициентов 
            b - вектор правой части
            transform - масив(вектор) с найденными коэффициентами афинного преобразования
            residuals - сумма квадратов остатков
            rank - ранг матрицы А
            s - сингулярные значения А
            rcond - порог для отбрасывания малых значений(Обычно по умолчанию ставим)
        '''
        return transform
    except  np.linalg.LinAlgError:
        print('Не удалось вычислить афинное проеобразование')
        return None
    


In [18]:
transform1=calculate_affine_transformation(src_coord, dst_coord)
print(transform1)

[ 0.94570487  0.0670213  -1.70632178  0.18210786  0.51697589 19.97612597]


In [ ]:
'''
 Про вывод: 
 
 изначально решаем x`=a*x+b*y+c
                 и y`=d*x+e*y+f
                 
 Тогда то что выводит наша функция: 
 это и есть a,b,c,d,e,f
 
 Где, а и e отвечают за масштаб(по x и y соотв), b и d отвечают за поворот влево-вправо по x и y соотв,
 с и f отвечают за смещение по x и y соотв(тогда это 3 и 6 значения, которые нам нужны)
 
'''

In [7]:
n=4
a=np.zeros((2*n,6))
b=np.zeros(2*n)
a

array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

In [8]:
b

array([0., 0., 0., 0., 0., 0., 0., 0.])

In [30]:
#применение полного афинного проеобразования к GeoDataFrame
def apply_affine_transformation(grid1, transform):
    
    a,b,c,d,e,f = transform
    
    #преобразование списка координат
    def transform_coords(coord):
        x,y = coord
        new_x = a*x+b*y+c
        new_y = d*x+e*y+f
        return(new_x,new_y)
    
    #делаем преобразование в зависисимости от типа геометрии(у нас у обоих файлов тип Polygon)
    def transform_geometry(geom):
        '''if geom.geom_type == 'Point':
            x,y=geom.coords[0]
            new_x = a*x+b*y+c
            new_y = d*x+e*y+f
            return Point(new_x,new_y)'''
        if geom.geom_type == 'Polygon':
            new_exter = [transform_coords(coord) for coord in geom.exterior.coords]
            new_inter = [[transform_coords(coord) for coord in interior.coords] for interior in geom.interiors]
            return Polygon(new_exter,*new_inter)
        else:
            raise ValueError(f'Необработанный типо геометрии {geom.geom.type}')
            
    grid1['geometry'] = grid1['geometry'].apply(transform_geometry)
    return grid1

In [31]:
transformed_grid1 = apply_affine_transformation(grid1.copy(), transform1)

In [32]:
transformed_grid1.to_file(output_file, driver='GeoJSON')

In [19]:
print(f'Смещение по X: {transform1[2]} ')
print(f'Смещение по Y: {transform1[5]} ')

Смещение по X: -1.706321780684019 
Смещение по Y: 19.97612597312276 


In [ ]:
''' 
Для чистоты эксперимента, проверим везде ли тип Polygon
(Другие типы геометрии при необходимости надо дописать)

'''

In [34]:
binary_string = ''.join(['1' if geom.geom_type == 'Polygon' else '0' for geom in grid1['geometry']])
print(binary_string)

...


In [37]:
print(binary_string.count('0'))

0


In [ ]:
#все сошлось, других типов геометрии нет